### Importando os Pacotes Utilitários
***

In [1]:
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow
import os

### Verificando as Configurações do Ambiente de Desenvolvimento
***

In [2]:
# verificando o diretório de desenvolvimento
HOME = os.getcwd()
print(HOME)

/home/alysson-barbosa/Documentos/esferas-cemig-visao-computacional/deteccao-e-segmentacao-torres/yolov8-deteccao


In [3]:
# verificando as configurações do CUDA para treinamento do modelo em GPU
!nvidia-smi

Thu Aug 31 11:38:08 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.43.04    Driver Version: 515.43.04    CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:01:00.0 Off |                  N/A |
| N/A   50C    P8     9W /  N/A |     10MiB /  4096MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [4]:
# verificando a compatibilidade do framework com o ambiente de desenvolvimento (CPU/GPU)
ultralytics.checks()

Ultralytics YOLOv8.0.152 🚀 Python-3.10.12 torch-2.0.1+cu117 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3910MiB)
Setup complete ✅ (16 CPUs, 31.1 GB RAM, 76.1/210.3 GB disk)


### Baixando o Conjunto de Imagens Pré-processado do RoboFLow
***

In [31]:
# baixando o conjunto de imagens do roboflow, utilizando a API integrada
# API_KEY = 'xhUBoRw9sdlgzMz1Zw0Y'
# WORKSPACE = 'alysson-machado'
# PROJECT = 'towers-instance-segmentation'
# FORMAT = 'yolov8'
# rf = Roboflow(api_key = API_KEY)
# project = rf.workspace(WORKSPACE).project(PROJECT)
# dataset = project.version(1).download(FORMAT)

### Preparando o Modelo para Treinamento

In [5]:
def define_version(directory, model_type = 'n'):
    '''
    Retorna o tipo de modelo da YOLOv8 de segmentação pré-treinado.

    Args:
        directory (str) --> diretório com os modelos salvos
        model_type (str) --> tipo do modelo (n, s, m, l, x)
    Return
        model_value (str) --> nome do modelo pré-treinado para treinamento
    '''

    return directory + 'yolov8' + model_type + '-seg.pt'

In [6]:
pretrained_model_dir = 'modelos-pretreinados-segmentacao/'
# instanciando o modelo treinável da YOLOv8
model = YOLO(define_version(pretrained_model_dir, 'm'))
print(model)

In [3]:
# === CONFIGURAÇÃO DO AMBIENTE DE TREINAMENTO ===

# salvar o modelo treinado ao decorrer das épocas
SAVE = True
# configuração do ambiente de treinamento (CUDA = 0, CPU = 1)
DEVICE = 0
# definindo os "worker threads" da GPU
WORKERS = 8
# utilizar o modelo pré-treinando carregado
PRETRAINED = True
# semente de geração do treinamento dos pesos para reprodutibilidade
SEED = 42
# validar o modelo após cada época de treinamento
VAL = True

### Treinamento do Modelo Instanciado
***

In [8]:
# === COMPILAÇÃO DO MODELO INSTANCIADO ===

# métrica de otimização do decrescimento da função de custo
# [SGD, Adam, Adamax, AdamW, NAdam, RAdam, RMSProp, auto]
OPTIMIZER = 'auto'
# taxas de aprendizado inicial e final (lr0*lr1) (i.e. SGD=1E-2, Adam=1E-3)
LR0, LRF = 1e-2, 1e-2
# taxa de momento do gradiente estocástico
MOMENTUM = 0.937
# taxa de dropout para inativação dos neurônios de saída
DROPOUT = 0

In [9]:
# === HIPER-PARÂMETROS DO MODELO ===

# arquivo de acesso ao conjunto de imagens
DATA = 'Towers-Instance-Segmentation-1/data.yaml'
# número de épocas de treinamento
EPOCHS = 100
# épocas de observação para ativação do "early stoppoing"
PATIENCE = 20  
# quantidade de lotes de imagens por etapa de treinamento
BATCH = -1
# dimensão das imagens de entrada
IMGSZ = 640

In [ ]:
# treinando o modelo instanciado
model.train(data = DATA, epochs = EPOCHS, patience = PATIENCE, batch = BATCH, 
            imgsz = IMGSZ, save = SAVE,device = DEVICE, workers = WORKERS, 
            pretrained = PRETRAINED, optimizer = OPTIMIZER, seed = SEED, 
            lr0 = LR0, lrf = LRF, momentum = MOMENTUM, dropout = DROPOUT, 
            val = VAL)

### Teste e Validação do Modelo Treinado
***

In [2]:
# === CONFIGURAÇÃO DOS PARÂMETROS DE PREDIÇÃO ===

# diretório de referência com a imagem ou vídeo para predição
SOURCE = 'video-example.mp4'
# limite de confiança do objeto para detecção
CONF = 0.25

In [ ]:
# realizando a predição no modelo treinado
model = YOLO('runs/segment/train5/weights/best.pt')
results = model.predict(source = SOURCE, conf = CONF, save = SAVE, stream=True)
for r in results:
    boxes = r.boxes
    masks = r.masks
    probs = r.probs